# Day-Trading Strategy Evidence Notebook

This notebook backs up the CEO strategy choice with tables and plots from the latest strategy backtest output.

It answers three board-level questions:

1. Which strategy families performed best?
2. Which ticker/strategy pairs justify tomorrow's paper-trading watchlist?
3. Why did we choose momentum breakout and opening-range breakout over VWAP reclaim/range reversion?

Run all cells in VS Code from the repo root. If you run a new backtest later, this notebook automatically loads the newest folder under `results/research/strategy_backtests`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

repo_root = Path.cwd()
if not (repo_root / "results" / "research" / "strategy_backtests").exists():
    # Allows running the notebook from notebooks/ as well as from repo root.
    repo_root = repo_root.parent

backtest_root = repo_root / "results" / "research" / "strategy_backtests"
runs = sorted([path for path in backtest_root.iterdir() if path.is_dir()])
assert runs, f"No backtest runs found under {backtest_root}"

run_dir = runs[-1]
summary_path = run_dir / "strategy_summary.csv"
aggregate_path = run_dir / "strategy_aggregate.csv"
trades_path = run_dir / "strategy_trades.csv"

summary = pd.read_csv(summary_path)
aggregate = pd.read_csv(aggregate_path)
trades = pd.read_csv(trades_path, parse_dates=["entry_time", "exit_time"])

print(f"Loaded backtest run: {run_dir}")
print(f"Summary rows: {len(summary):,}")
print(f"Trades: {len(trades):,}")
display(aggregate)

## Strategy Family Evidence

The first plot compares strategy families across the whole universe. The CEO choice should favor strategies with positive average total return, acceptable drawdown, and profit factor above 1.

In [ ]:
aggregate_ranked = aggregate.sort_values("avg_score", ascending=False).copy()

display(
    aggregate_ranked[[
        "strategy", "trades", "avg_win_rate_pct", "avg_total_return_pct",
        "avg_profit_factor", "avg_max_drawdown_pct", "avg_score"
    ]].style.format({
        "avg_win_rate_pct": "{:.2f}%",
        "avg_total_return_pct": "{:.2f}%",
        "avg_profit_factor": "{:.2f}",
        "avg_max_drawdown_pct": "{:.2f}%",
        "avg_score": "{:.2f}",
    })
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(aggregate_ranked["strategy"], aggregate_ranked["avg_score"], color="#3366AA")
axes[0].invert_yaxis()
axes[0].set_title("Average Strategy Score")
axes[0].set_xlabel("Score")

colors = ["#228B22" if value > 0 else "#B22222" for value in aggregate_ranked["avg_total_return_pct"]]
axes[1].barh(aggregate_ranked["strategy"], aggregate_ranked["avg_total_return_pct"], color=colors)
axes[1].axvline(0, color="black", linewidth=1)
axes[1].invert_yaxis()
axes[1].set_title("Average Total Return by Strategy")
axes[1].set_xlabel("Return %")

axes[2].barh(aggregate_ranked["strategy"], aggregate_ranked["avg_profit_factor"], color="#7A5195")
axes[2].axvline(1.0, color="black", linewidth=1, linestyle="--", label="breakeven PF")
axes[2].invert_yaxis()
axes[2].set_title("Average Profit Factor")
axes[2].set_xlabel("Profit factor")
axes[2].legend()

plt.tight_layout()
plt.show()

## Top Ticker/Strategy Pairs

This table is the direct evidence for choosing AMD and NVDA first. We require enough trades to matter, positive expectancy, profit factor above 1.15, and controlled drawdown.

In [ ]:
qualified = summary[
    (summary["trades"] >= 8)
    & (summary["total_return_pct"] > 0)
    & (summary["profit_factor"] >= 1.15)
    & (summary["max_drawdown_pct"] <= 5.0)
].sort_values("score", ascending=False)

top_pairs = qualified.head(16).copy()
display(
    top_pairs.style.format({
        "win_rate_pct": "{:.2f}%",
        "avg_return_pct": "{:.4f}%",
        "total_return_pct": "{:.2f}%",
        "profit_factor": "{:.2f}",
        "max_drawdown_pct": "{:.2f}%",
        "expectancy_pct": "{:.4f}%",
        "score": "{:.2f}",
    })
)

labels = top_pairs["ticker"] + "\n" + top_pairs["strategy"].str.replace("_", " ")
fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(labels, top_pairs["score"], color="#2F5597")
ax.set_title("Qualified Ticker/Strategy Pairs by Score")
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=65)
plt.tight_layout()
plt.show()

## Strategy/Ticker Heatmap

The heatmap shows where each strategy worked. Green cells are positive total return; red cells are negative. This is the clearest visual reason not to deploy broad VWAP reclaim or range reversion tomorrow.

In [ ]:
pivot = summary.pivot_table(
    index="ticker",
    columns="strategy",
    values="total_return_pct",
    aggfunc="mean",
).fillna(0)

# Sort tickers by their best strategy return so leaders rise to the top.
pivot = pivot.loc[pivot.max(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(15, 8))
vmax = max(abs(pivot.to_numpy()).max(), 1)
image = ax.imshow(pivot, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([col.replace("_", " ") for col in pivot.columns], rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title("Total Return % by Ticker and Strategy")

for row_idx, ticker in enumerate(pivot.index):
    for col_idx, strategy in enumerate(pivot.columns):
        value = pivot.loc[ticker, strategy]
        text_color = "white" if abs(value) > vmax * 0.45 else "black"
        ax.text(col_idx, row_idx, f"{value:.1f}", ha="center", va="center", color=text_color, fontsize=9)

fig.colorbar(image, ax=ax, label="Total return %")
plt.tight_layout()
plt.show()

## Risk/Reward Scatter

Good paper-trading candidates should sit toward the upper-left: higher profit factor, lower drawdown. AMD and NVDA momentum setups cluster where we want them.

In [ ]:
scatter = summary[summary["trades"] >= 8].copy()
fig, ax = plt.subplots(figsize=(13, 8))

for strategy, group in scatter.groupby("strategy"):
    ax.scatter(
        group["max_drawdown_pct"],
        group["profit_factor"],
        s=group["trades"] * 8,
        alpha=0.68,
        label=strategy.replace("_", " "),
    )

leaders = scatter.sort_values("score", ascending=False).head(10)
for _, row in leaders.iterrows():
    ax.annotate(
        f"{row['ticker']}\n{row['strategy'].replace('_', ' ')}",
        (row["max_drawdown_pct"], row["profit_factor"]),
        textcoords="offset points",
        xytext=(7, 5),
        fontsize=8,
    )

ax.axhline(1.15, color="black", linestyle="--", linewidth=1, label="deployment PF floor")
ax.axvline(5.0, color="gray", linestyle="--", linewidth=1, label="safe drawdown ceiling")
ax.set_title("Risk/Reward: Profit Factor vs Max Drawdown")
ax.set_xlabel("Max drawdown %")
ax.set_ylabel("Profit factor")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## Equity Curves For Chosen Setups

This plot compounds trade returns for the highest-ranked ticker/strategy pairs. It is not live P/L; it is a normalized backtest equity curve showing how consistent each setup was across recent trading days.

In [ ]:
chosen_pairs = [
    ("AMD", "opening_range_breakout_15m"),
    ("AMD", "momentum_breakout"),
    ("NVDA", "momentum_breakout"),
    ("NVDA", "relative_strength_continuation"),
    ("COIN", "momentum_breakout"),
    ("INTC", "momentum_breakout"),
]

fig, ax = plt.subplots(figsize=(14, 7))
curve_table = []
for ticker, strategy in chosen_pairs:
    subset = trades[(trades["ticker"] == ticker) & (trades["strategy"] == strategy)].sort_values("exit_time")
    if subset.empty:
        continue
    equity = (1 + subset["return_pct"].astype(float) / 100.0).cumprod()
    label = f"{ticker} {strategy.replace('_', ' ')}"
    ax.plot(subset["exit_time"], equity, marker="o", markersize=3, linewidth=1.7, label=label)
    curve_table.append({
        "ticker": ticker,
        "strategy": strategy,
        "trades": len(subset),
        "final_equity_multiple": equity.iloc[-1],
        "total_return_pct": (equity.iloc[-1] - 1) * 100,
    })

ax.axhline(1.0, color="black", linewidth=1)
ax.set_title("Normalized Equity Curves For Selected Setups")
ax.set_ylabel("Equity multiple")
ax.set_xlabel("Trade exit time")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

display(pd.DataFrame(curve_table).sort_values("total_return_pct", ascending=False).style.format({
    "final_equity_multiple": "{:.4f}",
    "total_return_pct": "{:.2f}%",
}))

## Tomorrow's Strategy Choice

Based on the evidence above, the plan is:

- Safe profile: prioritize `AMD` and `NVDA`, using `opening_range_breakout_15m` and selective `relative_strength_continuation` only when live spread, volume, and price gates confirm.
- Risky profile: prioritize `AMD`, `NVDA`, `COIN`, and `INTC`, using `momentum_breakout`, `opening_range_breakout_15m`, and selective `relative_strength_continuation`.
- Exclude broad VWAP reclaim and range reversion tomorrow because their aggregate returns and profit factors were weak.

The live runner still has final authority at market open: it must confirm Alpaca market-open status, fresh latest trade, clean spread, volume confirmation, and bracket stop/take-profit before any paper order is submitted.